# SpatialQuery: query-driven analysis of multicellular spatial motifs
> We recommend setting the memory on the workspace advanced configuration settings to at least 64GB since correlation analyses can be memory-consuming.

Spatial transcriptomics and spatial proteomics technologies profile gene or protein expression while preserving spatial coordinates, enabling analysis of how cells organize and interact in tissues. A key challenge is to systematically identify recurring multicellular spatial patterns and characterize their molecular consequences.

SpatialQuery addresses this by providing a unified framework centered on the concept of **cellular motifs** — recurring combinations of cell types that co-localize in spatial neighborhoods of a given anchor cell type. The framework supports two interconnected analytical modules:

1. **Motif discovery and enrichment analysis**: Given an anchor cell type, SpatialQuery identifies frequently co-occurring cell type combinations in its spatial neighborhood via frequent pattern mining (FP-Growth), and tests whether these motifs are enriched in the neighborhood of anchor cell type beyond chance using hypergeometric testing.

2. **Motif-associated molecular characterization**: For a discovered motif, SpatialQuery performs motif-associated differential expression analysis between cells surrounded by the motif and those that do not, and cross-cell covariation analysis within the motif to reveal molecular programs linked to specific spatial contexts.

The input is any cell-type-annotated spatial transcriptomics or spatial proteomics dataset with spatial coordinates.

### Single Dataset Analysis

1. Motif discovery: identification of frequent cell type co-localization patterns across the tissue or around a specified anchor cell type
2. Motif enrichment analysis: statistical testing of motif significance surrounding the anchor cell type
3. Motif-associated differential expression analysis between anchor cells surrounded by motif and those lacking it 
4. Motif-associated cross-cell gene–gene covariation analysis between anchor cells and motif cells
5. Interactive visualization through Vitessce

### Multiple Dataset Analysis

1. Integrative motif discovery across multiple fields of view
2. Integrative motif enrichment analysis across datasets
3. Differential motif enrichment analysis between conditions (e.g., healthy vs. disease)
4. Motif-associated differential expression analysis and cross-cell gene-gene covariation analysis

SpatialQuery achieves real-time response for individual datasets and completes atlas-scale analyses (millions of cells) within minutes, enabled by k-D tree spatial indexing and closed-form statistical tests that avoid iterative optimization or permutation-based inference.

In [ ]:
!pip install SpatialQuery zarr tqdm vitessce esbuild_py uvicorn starlette oxc_py anywidget hubmap-template-helper

In [ ]:
import requests
import json
import os
import warnings

import numpy as np
import pandas as pd
import anndata as ad
import zarr

from tqdm import tqdm

from hubmap_template_helper import compatibility as hth_comp

from SpatialQuery import spatial_query
from SpatialQuery import spatial_query_multi
from SpatialQuery import interactive_motif
from SpatialQuery.plotting import (
    plot_fp_heatmap,
    plot_motif_enrichment_heatmap,
    plot_differential_pattern_heatmap,
)

warnings.filterwarnings("ignore")
pd.set_option('display.max_colwidth', 1000)
pd.set_option('display.max_columns', 500)

In [ ]:
# linked datasets
uuids = {{ uuids | safe }}

In [ ]:
search_api = 'https://search.api.hubmapconsortium.org/v3/portal/search'

accepted_assay_display_names = ["Slide-seq [Salmon]"]

In [ ]:
print(len(uuids))
uuids = hth_comp.check_template_compatibility(uuids, search_api=search_api, accepted_assay_display_names=accepted_assay_display_names)
print(len(uuids))

# Load spatial transcriptomics data
The following datasets were symlinked to the workspace when this template was added. 

> We have tested SpatialQuery primarily with [Slide-seq datasets](https://portal.hubmapconsortium.org/search?raw_dataset_type_keyword-assay_display_name_keyword[Slide-seq][0]=Slideseq&raw_dataset_type_keyword-assay_display_name_keyword[Slide-seq][1]=Slideseq%20%5BSalmon%5D&entity_type[0]=Dataset) which include cell type annotations.

In [ ]:
adatas = []
adata_zarr_paths = [] # for vitessce
for uuid in tqdm(uuids):
    adata_uuid = ad.read_h5ad('datasets/' + uuid + '/secondary_analysis.h5ad')
    adata_uuid.X = adata_uuid.layers['spliced'].copy()  # Set adata.X with count data
    adatas.append(adata_uuid)
    adata_zarr_paths.append('datasets/' + uuid + '/hubmap_ui/anndata-zarr/secondary_analysis.zarr')

In [ ]:
num_cells = np.sum([adata.n_obs for adata in adatas])
print(f"Number of total cells: {num_cells}")

# Single Field of View (FOV) Analysis by SpatialQuery

This section demonstrates the application of SpatialQuery for analyzing a single field of view (FOV) in spatial transcriptomics data. The process begins with the initialization of a SpatialQuery object, which involves constructing a KD-tree using spatial location data and storing labels for each spot.

In this notebook, we utilize an annotated AnnData object loaded from "secondary_analysis.h5ad". The key components for initialization are:

1. **adata**: Annotated data object containing expression matrix and metadata.
2. **dataset**: An optional dataset identifier to uniquely name each FOV.
3. **spatial_key**: Key in `adata.obsm` containing spatial coordinates.
4. **label_key**: Key in `adata.obs` containing cell type annotations.
5. **leaf_size**: Leaf size parameter for KDTree construction. Larger values reduce tree depth but increase computation per query.
6. **build_gene_index**: If True, constructs SCFind index by compressing expression data to binary format to save memory. If False, uses the original expression matrix from `adata.X` directly.
7. **feature_name**: Column name in `adata.var` containing feature (gene) identifiers.
8. **if_lognorm**: If True, applies log1p transformation after library size normalization. Set to False if expression data is already normalized.
9. **if_normalize_spatial_coord**: If True, normalizes spatial coordinates so mean nearest neighbor distance equals 1, making `max_dist` interpretable as "number of cell diameters." Set to False to preserve original spatial units (e.g., micrometers).


In [ ]:
spatial_key = 'X_spatial'
label_key = 'predicted_label'
feature_name='hugo_symbol'

adata = adatas[0]

single_sp = spatial_query(
    adata=adata,
    dataset="single-fov",
    spatial_key=spatial_key,
    label_key=label_key,
    leaf_size=10,
    build_gene_index=False,
    feature_name=feature_name,
    if_lognorm=True,
    if_normalize_spatial_coord=True,
)


### Visualizaiton of the Spatial Distribution of Cell Types by plot_fov

In [ ]:
single_sp.plot_fov(
    min_cells_label=20,  # displaying cell types more than 20 cells
    title='Spatial distribution of cell types', 
    figsize=(10, 5)
)

### Identification of Frequent Patterns of Cell Types in Single FOV with find_patterns/rand
In the absence of prior knowledge about the dataset, the find_patterns_grid and find_patterns_rand methods can be employed to identify frequent patterns of cell types in cellular neighborhoods across whole FOV. These functions serve as valuable tools for preliminary data exploration, offering insights into the spatial organization of different cell types within the tissue.

In [ ]:
fp_grid = single_sp.find_patterns_grid(
    max_dist=10,      # the radius of neighborhood to be considered
    min_size=0,        # minimum number of cells allowed in a neighborhood
    min_support=0.5,   # lower bound of the support value for the frequency of each motif
    if_display=True,   # whether to display spatial distribution of frequent motifs of cell types
    figsize=(6, 5),   # customize size of the output figure
    return_cellID=False, # whether to return the IDs of neighboring cells in frequent motifs
)

In [ ]:
fp_grid

The `plot_fp_heatmap` function provides a heatmap visualization of frequent patterns, displaying the frequency for each motif

In [ ]:
single_sp.plot_fp_heatmap(fp_grid, figsize=(10, 5))

In [ ]:
# Or use randomly selected points as the center of neighborhood to find frequent patterns of cell types.
fp_rand = single_sp.find_patterns_rand(
    max_dist=10.0,
    n_points=1000,     # number of randomply selected points
    min_support=0.5,
    min_size=0,
    if_display=True,
    figsize=(9, 5),
    seed=2024
)
fp_rand.head(5)

In addition to visualizing the spatial distribution of all cells in frequent patterns, SpatialQuery also supports the visualization of spatial distributions for individual motifs. This feature allows for a more detailed examination of specific pattern distribution within the tissue.

In [ ]:
for motif in fp_grid['itemsets'].head(3):
    single_sp.plot_motif_grid(
        motif=motif,
        figsize=(9,5),
        max_dist=10.0
    )

### Identification of Frequent Patterns Around an Anchor Cell Type of Interest with find_fp_knn/dist

For researchers interested in exploring the microenvironment surrounding a specific cell type, SpatialQuery offers targeted analysis capabilities. Users can specify a central cell type of interest as anchors and subsequently identify frequent patterns in its neighborhood using either k-nearest neighbors (kNN) or radius-based approaches. This functionality allows for the detailed examination of cellular contexts and potential interactions specific to the chosen cell type.

The analysis can be performed as follows:

1. Specify the cell type of interest
2. Choose between kNN or radius-based neighborhood definition
3. Set appropriate parameters (k value for kNN or radius for radius-based approach)
4. Execute the pattern identification algorithm

In the following usecase, podocyte is studied as the central cell type. You can set this to other cell types that are present in the datasets.

In [ ]:
central_ct = 'podocyte'
fp_knn = single_sp.find_fp_knn(
    ct=central_ct,
    k=30,
    min_support=0.5
)
fp_knn.head(5)

In [ ]:
# Same functionality but using radius-based neighborhood
central_ct = 'podocyte'
fp_dist = single_sp.find_fp_dist(
    ct=central_ct,
    max_dist=10,
    min_support=0.5,
    min_size=0,
)
fp_dist.head(5)

We can also visualize the frequent patterns around the central cell type as a heatmap.

In [ ]:
single_sp.plot_fp_heatmap(fp_knn, 
                          title=f'Frequent patterns around {central_ct} (KNN)',
                         figsize=(15, 5))

### Motif Enrichment Analysis Around Anchor Cell Type with motif_enrichment_knn/dist

SpatialQuery identifies statistically significant motifs in the spatial neighborhood of a given anchor cell type. It first discovers frequent patterns via `find_fp_knn`/`find_fp_dist`, then tests whether each motif is enriched beyond chance using a hypergeometric test. Alternatively, users can specify a motif of interest directly and test its enrichment. The functions `motif_enrichment_knn` and `motif_enrichment_dist` support k-nearest neighbor and distance-based neighborhood definitions, respectively.


In [ ]:
motif_sig_knn = single_sp.motif_enrichment_knn(
    ct=central_ct,
    k=30,
    min_support=0.5,
)

motif_sig_knn.head(3)


In [ ]:
# Same analysis but with radius-based neighborhood
motif_sig_dist = single_sp.motif_enrichment_dist(
    ct=central_ct,
    max_dist=10,
    min_support=0.5,
)

motif_sig_dist.head(3)


The `plot_motif_enrichment_heatmap` function visualizes the enrichment analysis results, showing significance levels and enrichment scores for each motif.

In [ ]:
single_sp.plot_motif_enrichment_heatmap(motif_sig_knn.head(5), 
                                        title=f'Motif enrichment around {central_ct} (KNN)',
                                        figsize=(10, 5)
                                       )

#### Enrichment Analysis with Specified Motifs

Users can also specify motifs of interest — either from prior knowledge or from the frequent pattern analysis above — to test their enrichment explicitly.


In [ ]:
# Specify a motif from the frequent pattern results
motif = list(fp_knn['itemsets'][0])
motif_sig_custom = single_sp.motif_enrichment_knn(
    ct=central_ct,
    motifs=motif,
    k=30,
)

motif_sig_custom


### Visualization of Specified Cell Types around Central Cell Type with plot_motif_celltype
Moreover, SpatialQuery supports the visualization of specified patterns around a central cell type using the plot_motif_celltype function. This feature enables researchers to visually inspect the spatial distribution of particular cellular arrangements within the tissue context. The red circle denotes the central cell type while the colorful dots correspond to neighboring cell types in given pattern.

In [ ]:
motif = motif_sig_custom['motifs'][0]
single_sp.plot_motif_celltype(
    ct=central_ct, 
    motif=motif, 
    figsize=(5, 5)
)

## Differential Expression Analysis with de_genes

SpatialQuery supports differential expression analysis between two groups. Here, we compared anchor cells surrounded by a given motif and those lacking it, revealing genes whose expression differs associated with the presence of specific multicellular spatial contexts.

In [ ]:
motif = motif_sig_custom['motifs'][0]
motif_result_dist = single_sp.motif_enrichment_dist(
    ct=central_ct,
    motifs=motif,
    max_dist=10,
    return_cellID=True,
)

center_id = motif_result_dist["center_id"].iloc[0]
all_center_id = np.where(single_sp.labels == central_ct)[0]
non_center_id = np.setdiff1d(all_center_id, center_id)

print(f"Motif+ anchor cells: {len(center_id)}")
print(f"Motif− anchor cells: {len(non_center_id)}")

In [ ]:
de_result = single_sp.de_genes(
    ind_group1=center_id,
    ind_group2=non_center_id,
    min_fraction=0.05,
    method="t-test",
    alpha=0.05,
)

print(f"Number of DE genes: {len(de_result)}")
de_result.head(10)

## Gene–Gene Covariation Analysis with compute_gene_gene_correlation / compute_gene_gene_correlation_by_type

SpatialQuery performs cross-cell gene–gene covariation analysis between anchor cells and their motif neighbors, comparing correlations in motif-proximal cells against non-proximal and non-motif background to identify gene pairs with motif-specific coordinated expression changes. Two functions are provided: `compute_gene_gene_correlation` pools all motif constituent cell types together for a unified analysis, while `compute_gene_gene_correlation_by_type` computes covariation separately for each cell type within the motif relative to the anchor cells.

To speed up computation of whole transcriptomics data, we recommend selecting highly variable genes for most informative features.

In [ ]:
import scanpy as sc
adata_tmp = single_sp.adata.copy()
sc.pp.highly_variable_genes(adata_tmp, n_top_genes=3000)
hvg = adata_tmp.var[adata_tmp.var['highly_variable']].index.tolist()

In [ ]:
motif = list(fp_knn['itemsets'][0])
gene_pair_df = single_sp.compute_gene_gene_correlation_by_type(
    ct=central_ct,
    motif=motif,
    genes=hvg,  # specify genes to compute gene-gene correlation, or use all genes by setting genes=None
    max_dist=10,  # define neighborhood size with radius-based neighborhood, or specify k for knn-based neighborhood
)

gene_pair_df = gene_pair_df[gene_pair_df['if_significant']]
gene_pair_df.value_counts('cell_type')

In [ ]:
gene_pair_df.head(10)

#### Visualization of Gene-Gene Correlations with plot_gene_pair_heatmap

The `plot_gene_pair_heatmap` function displays a heatmap of gene-gene correlation scores.

In [ ]:
single_sp.plot_gene_pair_heatmap(gene_pair_df)

In [ ]:
import matplotlib.pyplot as plt
top_n = 20
cell_types = gene_pair_df["cell_type"].unique()

for ct in cell_types:
    ct_df = gene_pair_df[gene_pair_df["cell_type"] == ct]
    if len(ct_df) == 0:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Neighbor cell type: {ct}", fontsize=14)

    ct_df["gene_center"].value_counts().head(top_n).plot.barh(
        ax=axes[0], color="steelblue")
    axes[0].set_title(f"Top {top_n} Anchor Genes (gene_center)")
    axes[0].set_xlabel("Number of significant pairs")
    axes[0].invert_yaxis()

    ct_df["gene_motif"].value_counts().head(top_n).plot.barh(
        ax=axes[1], color="coral")
    axes[1].set_title(f"Top {top_n} Motif Genes (gene_motif)")
    axes[1].set_xlabel("Number of significant pairs")
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

## Interactive Visualization with interactive_motif

SpatialQuery provides `interactive_motif`, a one-line function to launch an interactive Vitessce widget for exploring spatial motifs in Jupyter notebooks. The widget includes spatial scatter plots, cell type browsers, and an interactive query panel supporting motif discovery as listed above for real-time motif enrichment analysis.

> Interactive visualization currently only supports the single-FOV analysis case.

In [ ]:
interactive_motif(single_sp, zarr_path='./interactive_motif.zarr')

# Integrative Analysis of Multiple FOVs

SpatialQuery extends the above single-dataset analyses to multiple FOVs, enabling integrative motif discovery, enrichment analysis, and differential motif enrichment analysis across samples or experimental conditions. To distinguish between FOVs, users should specify unique, indicative dataset names for each FOV during initialization.

For illustrative purposes, we create simulated dataset names representing "normal" and "disease" states to showcase the workflow for comparing patterns between different biological conditions.

In [ ]:
n_datasets = len(adatas)
half = round(n_datasets / 2)
datasets_name = ['normal'] * half + ['disease'] * (len(adatas) - half)
multi_sp = spatial_query_multi(
    adatas=adatas,  
    datasets=datasets_name,
    spatial_key=spatial_key, 
    label_key=label_key,
    build_gene_index=False,
    feature_name=feature_name,
    if_lognorm=True,
    if_normalize_spatial_coord=True,
    leaf_size=10
)

### Identification of Frequent Patterns of Cell Types accross Multiple FOVs with find_fp_knn/dist
SpatialQuery provides functions find_fp_knn and find_fp_dist for identifying frequent patterns of cell types across multiple FOVs. These functions are analogous to their single-FOV counterparts. Users can explicitly define which datasets (FOVs) to include in the analysis. If not specified, the functions will, by default, use all available datasets.

In [ ]:
central_ct = "podocyte"
fp_knn_multi = multi_sp.find_fp_knn(
    ct=central_ct, 
    dataset='normal',
    k=30,
    min_support=0.5,
    max_dist=100
)

fp_knn_multi.head(5)

In [ ]:
# Radius-based neighborhoods
central_ct = "podocyte"
fp_dist_multi = multi_sp.find_fp_dist(
    ct=central_ct, 
    dataset='normal',
    max_dist=10,
    min_support=0.5,
)

fp_dist_multi.head(5)

### Enrichment Analysis of Patterns Around Central Cell Type across Multiple FOVs with motif_enrichment_knn/dist
SpatialQuery also extends its enrichment analysis capabilities to multiple FOVs, employing the same methodological approach as used in single FOV analysis.

In [ ]:
# Enrichment analysis for specified motif with KNN-based neighborhoods
motif = fp_knn_multi['itemsets'][0]
motif_sig_knn_multi = multi_sp.motif_enrichment_knn(
    ct=central_ct,
    motifs=motif, 
    dataset='normal'
)

motif_sig_knn_multi

In [ ]:
# Enrichment analysis for specified motif with radius-based neighborhoods
motif = fp_dist_multi['itemsets'][0]
motif_sig_dist_multi = multi_sp.motif_enrichment_dist(
    ct=central_ct,
    motifs=motif, 
    dataset='normal'
)

motif_sig_dist_multi

## Differential Motif Enrichment Analysis across Conditions

For multiple FOVs representing different conditions, SpatialQuery performs differential motif enrichment analysis to identify motifs that are significantly enriched or depleted around a given anchor cell type in one condition relative to another.

In [ ]:
out_knn = multi_sp.differential_analysis_knn(
    ct=central_ct,
    datasets=['normal', 'disease'],
    k=20,
    min_support=0.3,
    max_dist=100,
)
out_knn['normal'].head(5)

In [ ]:
out_knn['disease'].head(5)

Since there's no biological difference between the two conditions by randomly separating the hemogeneous datasets, we expect no motifs to be specifically enriched in the disease dataset relative to the normal dataset. In application of real-world data, users can use `plot_differential_pattern_heatmap` to visualize the differential pattern analysis results showing condition-specific enriched patterns:
`plot_differential_pattern_heatmap(out_knn, ct=central_ct)`

We can repeat this same differential analysis with a radius-based neighborhood.

In [ ]:
out_dist = multi_sp.differential_analysis_dist(
    ct=central_ct,
    datasets=['normal', 'disease'],
    max_dist=10,
    min_support=0.3,
)
out_dist['normal'].head(5)

In [ ]:
out_dist['disease'].head(5)

## Differential Expression Analysis across FOVs

SpatialQuery extends differential expression analysis to the multi-FOV setting via `de_genes`. Cell groups can be defined flexibly across FOVs to support different comparison designs: for example, motif-surrounded versus non-surrounded anchor cells pooled across FOVs within the same condition, or motif+ anchor cells compared between conditions to identify condition-specific transcriptional programs associated with a given spatial context.

Use `motif_enrichment_dist`/`motif_enrichment_knn` with `return_cellID=True` to obtain the indices of motif-positive anchor cells and their neighbors for each condition. The returned cell IDs are organized as dictionaries keyed by FOV name.

### DE analysis for motif+ anchors between conditions

In [ ]:
motif = motif_sig_custom['motifs'][0]
control_result, control_motif_id, control_center_id = multi_sp.motif_enrichment_dist(
    ct=central_ct, motifs=motif, dataset='normal', 
    max_dist=10, return_cellID=True,
)

case_result, case_motif_id, case_center_id = multi_sp.motif_enrichment_dist(
    ct=central_ct, motifs=motif, dataset='disease', 
    max_dist=10, return_cellID=True,
)

print(f"Control motif+ anchors: {sum(len(v) for v in control_center_id[str(sorted(motif))].values())} cells")
print(f"Case motif+ anchors: {sum(len(v) for v in case_center_id[str(sorted(motif))].values())} cells")

In [ ]:
de_across = multi_sp.de_genes(
    ind_group1=control_center_id[str(sorted(motif))],
    ind_group2=case_center_id[str(sorted(motif))],
    method="t-test",
    alpha=0.05,
)

print(f"DE genes (Control vs Case motif+ {central_ct}): {len(de_across)}")
de_across.head(10)

### DE analysis for motif+ vs motif− anchors within one condition

In [ ]:
# Get non-motif anchor cell IDs for disease FOVs
non_motif_center = {str(sorted(motif)): {}}
for sp in multi_sp.spatial_queries:
    if sp.dataset.split("_")[0] != 'disease':
        continue
    ct_id = np.where(sp.labels == central_ct)[0]
    motif_ids = case_center_id[str(sorted(motif))].get(sp.dataset, [])
    non_motif_center[str(sorted(motif))][sp.dataset] = list(set(ct_id) - set(motif_ids))
    
de_within = multi_sp.de_genes(
    ind_group1=case_center_id[str(sorted(motif))],
    ind_group2=non_motif_center[str(sorted(motif))],
    method="t-test",
    alpha=0.05,
)

print(f"DE genes (motif+ vs motif− in disease): {len(de_within)}")
de_within.head(10)

## Gene–Gene Covariation Analysis across FOVs

SpatialQuery extends cross-cell gene–gene covariation analysis to the multi-FOV setting, pooling anchor and motif neighbor cells across FOVs to identify motif-specific coordinated expression patterns with greater statistical power. As in the single-dataset case, `compute_gene_gene_correlation` pools all motif constituent cell types for a unified covariation analysis, while `compute_gene_gene_correlation_by_type` computes covariation separately for each cell type within the motif relative to the anchor cells.

For whole-transcriptomic data, we recommend pre-selecting highly variable genes (HVGs) speeds up computation and focuses the analysis on the most informative features.

In [ ]:

# Select top 3000 HVGs per condition and take the union
import scanpy as sc

tt1 = np.array([f.replace('_', '-') for f in datasets_name])
tt2 = []
for adata in adatas:
    adata.var_names = adata.var[feature_name]
    # Drop features with NaN names
    adata = adata[:, adata.var_names.notna()].copy()
    # Remove duplicated features
    adata = adata[:, ~adata.var_names.duplicated()].copy()
    tt2.append(adata)

selected_genes = {}
for ds in ['normal', 'disease']:
    mask = np.where(tt1 == ds)[0]
    adata_sub = ad.concat([tt2[i].copy() for i in mask], join='inner')
    print(f"{ds}: {adata_sub.shape}")
    sc.pp.normalize_total(adata_sub)
    sc.pp.log1p(adata_sub)
    sc.pp.highly_variable_genes(adata_sub, n_top_genes=3000)
    selected_genes[ds] = adata_sub.var[adata_sub.var['highly_variable']].index.tolist()

In [ ]:
motif = list(fp_knn['itemsets'][0])
covarying = multi_sp.compute_gene_gene_correlation_by_type(
    ct=central_ct,
    motif=motif,
    dataset='disease',
    max_dist=10,
    genes=selected_genes['disease'],  # restrict to selected HVGs
    alpha=0.05,
)

covarying_sig = covarying[covarying["if_significant"]].copy()
print(f"Total gene pairs tested: {len(covarying)}")
print(f"Significant covarying pairs: {len(covarying_sig)}")

gene_pair_df.value_counts('cell_type')

For `compute_gene_gene_correlation_by_type`, visualize the top 20 genes separately for each neighbor cell type.

In [ ]:
import matplotlib.pyplot as plt
top_n = 20
cell_types = covarying_sig["cell_type"].unique()

for ct in cell_types:
    ct_df = covarying_sig[covarying_sig["cell_type"] == ct]
    if len(ct_df) == 0:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Neighbor cell type: {ct}", fontsize=14)

    ct_df["gene_center"].value_counts().head(top_n).plot.barh(
        ax=axes[0], color="steelblue")
    axes[0].set_title(f"Top {top_n} Anchor Genes (gene_center)")
    axes[0].set_xlabel("Number of significant pairs")
    axes[0].invert_yaxis()

    ct_df["gene_motif"].value_counts().head(top_n).plot.barh(
        ax=axes[1], color="coral")
    axes[1].set_title(f"Top {top_n} Motif Genes (gene_motif)")
    axes[1].set_xlabel("Number of significant pairs")
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

In [ ]:
covarying_sig["cell_type"].value_counts()